# 06 — Alice Image to 3D Model Generation with Tripo API

This notebook prepares an API-based workflow for converting Alice-related reference images into 3D models.

The workflow follows this structure:

1. Load reference images
2. Upload images to Tripo
3. Create image-to-3D tasks
4. Poll task status
5. Download generated 3D model files

The output models are saved as OBJ files and used in the next notebook:

`07_obj_to_pointcloud_batch.ipynb`

In [ ]:
import os
import time
import json
import requests
from pathlib import Path

import pandas as pd

## 1. Path and API settings

Reference images should be placed in:

`data/raw/alice_reference_images/`

Generated OBJ models will be saved in:

`data/raw/alice_models/`

The API key should not be written directly into the notebook. It should be stored as an environment variable.

In [ ]:
# =========================
# Path settings
# =========================

IMAGE_DIR = Path("../../data/raw/alice_reference_images")
MODEL_DIR = Path("../../data/raw/alice_models")

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# API settings
# =========================

TRIPO_API_KEY = os.getenv("TRIPO_API_KEY")

if TRIPO_API_KEY is None:
    raise ValueError(
        "TRIPO_API_KEY is not set. "
        "Set it as an environment variable before running this notebook."
    )

BASE_URL = "https://api.tripo3d.ai/v2/openapi"

HEADERS = {
    "Authorization": f"Bearer {TRIPO_API_KEY}"
}

print("Image folder:", IMAGE_DIR.resolve())
print("Model folder:", MODEL_DIR.resolve())

## 2. Collect reference images

The images should be clean, centred, and contain one main object.

Supported formats:

- `.png`
- `.jpg`
- `.jpeg`
- `.webp`

Each image should correspond to one Alice-related object, for example:

- `rabbit.png`
- `alice.png`
- `cat.png`
- `hat.png`
- `tea.png`

In [ ]:
image_files = sorted([
    f for f in IMAGE_DIR.iterdir()
    if f.suffix.lower() in [".png", ".jpg", ".jpeg", ".webp"]
])

print(f"Found {len(image_files)} reference images:")

for image_path in image_files:
    print("-", image_path.name)

## 3. Upload image to Tripo

This function uploads one local reference image to Tripo.

The returned image token / file id will be used to create an image-to-3D task.

In [ ]:
def upload_image(image_path: Path) -> dict:
    """
    Upload one image file to Tripo.

    Returns the API response as a dictionary.
    """
    upload_url = f"{BASE_URL}/upload"

    with open(image_path, "rb") as f:
        files = {
            "file": (image_path.name, f, "application/octet-stream")
        }

        response = requests.post(
            upload_url,
            headers=HEADERS,
            files=files,
            timeout=120
        )

    if response.status_code != 200:
        raise RuntimeError(
            f"Upload failed for {image_path.name}\n"
            f"Status: {response.status_code}\n"
            f"Response: {response.text}"
        )

    return response.json()

## 4. Create image-to-3D task

This function creates a 3D generation task from an uploaded image.

Depending on the current Tripo API version, the exact image key may differ.

If the API response structure changes, inspect the upload response first and update `image_token` accordingly.

In [ ]:
def create_image_to_3d_task(image_token: str) -> dict:
    """
    Create an image-to-3D task.

    The payload may need minor adjustment depending on Tripo's current API schema.
    """
    task_url = f"{BASE_URL}/task"

    payload = {
        "type": "image_to_model",
        "file": {
            "type": "image",
            "file_token": image_token
        },
        "model_version": "v2.5",
        "face_limit": 2500
    }

    response = requests.post(
        task_url,
        headers={**HEADERS, "Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=120
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Task creation failed\n"
            f"Status: {response.status_code}\n"
            f"Response: {response.text}"
        )

    return response.json()

## 5. Poll task status

3D generation is asynchronous.

This function repeatedly checks the task until it finishes or fails.

In [ ]:
def poll_task(task_id: str, interval: int = 10, max_wait: int = 900) -> dict:
    """
    Poll a Tripo task until it completes or fails.
    """
    task_url = f"{BASE_URL}/task/{task_id}"

    start_time = time.time()

    while True:
        response = requests.get(
            task_url,
            headers=HEADERS,
            timeout=120
        )

        if response.status_code != 200:
            raise RuntimeError(
                f"Task polling failed\n"
                f"Status: {response.status_code}\n"
                f"Response: {response.text}"
            )

        result = response.json()
        status = result.get("data", {}).get("status") or result.get("status")

        print("Task status:", status)

        if status in ["success", "succeeded", "completed"]:
            return result

        if status in ["failed", "error"]:
            raise RuntimeError(f"Task failed: {result}")

        if time.time() - start_time > max_wait:
            raise TimeoutError(f"Task timeout after {max_wait} seconds")

        time.sleep(interval)

## 6. Download model file

This function downloads the generated model file.

For this project, OBJ is preferred because it can be converted into point clouds in the next notebook.

In [ ]:
def download_file(file_url: str, output_path: Path):
    """
    Download a generated model file from a URL.
    """
    response = requests.get(file_url, timeout=300)

    if response.status_code != 200:
        raise RuntimeError(
            f"Download failed\n"
            f"Status: {response.status_code}\n"
            f"Response: {response.text[:500]}"
        )

    with open(output_path, "wb") as f:
        f.write(response.content)

    print("Saved:", output_path)

## 7. Batch process reference images

This cell loops through all reference images and generates corresponding 3D models.

Important:

- This cell may use paid API credits.
- Run it only after checking the image list.
- If the API response structure changes, inspect the printed responses and update the token / URL extraction.

In [ ]:
generated_records = []

for image_path in image_files:
    object_name = image_path.stem
    output_obj_path = MODEL_DIR / f"{object_name}.obj"

    if output_obj_path.exists():
        print(f"Skipping existing model: {output_obj_path.name}")
        continue

    print("=" * 60)
    print("Processing:", image_path.name)

    # 1. Upload image
    upload_response = upload_image(image_path)
    print("Upload response:")
    print(json.dumps(upload_response, indent=2)[:1000])

    # You may need to adjust this line after checking the real upload response
    image_token = (
        upload_response.get("data", {}).get("file_token")
        or upload_response.get("data", {}).get("image_token")
        or upload_response.get("file_token")
    )

    if image_token is None:
        raise KeyError(
            "Could not find image token in upload response. "
            "Please inspect upload_response and update the extraction logic."
        )

    # 2. Create task
    task_response = create_image_to_3d_task(image_token)
    print("Task response:")
    print(json.dumps(task_response, indent=2)[:1000])

    task_id = (
        task_response.get("data", {}).get("task_id")
        or task_response.get("task_id")
        or task_response.get("id")
    )

    if task_id is None:
        raise KeyError(
            "Could not find task_id in task response. "
            "Please inspect task_response and update the extraction logic."
        )

    # 3. Poll task
    final_response = poll_task(task_id)
    print("Final response:")
    print(json.dumps(final_response, indent=2)[:1500])

    # You may need to adjust this line after checking the real final response
    model_url = (
        final_response.get("data", {}).get("output", {}).get("model")
        or final_response.get("data", {}).get("output", {}).get("obj")
        or final_response.get("data", {}).get("result", {}).get("model")
        or final_response.get("data", {}).get("model_url")
    )

    if model_url is None:
        raise KeyError(
            "Could not find model download URL in final response. "
            "Please inspect final_response and update the extraction logic."
        )

    # 4. Download OBJ
    download_file(model_url, output_obj_path)

    generated_records.append({
        "object_name": object_name,
        "image_path": str(image_path),
        "model_path": str(output_obj_path),
        "task_id": task_id
    })

print("Batch generation finished.")

## 8. Save generation log

This log records which reference images were used to generate which model files.

In [ ]:
log_path = MODEL_DIR / "alice_model_generation_log.csv"

if generated_records:
    log_df = pd.DataFrame(generated_records)
    log_df.to_csv(log_path, index=False)
    display(log_df)
    print("Log saved:", log_path)
else:
    print("No new models were generated in this run.")